In [1]:
!pip install pymanopt autograd


   -------------------- ------------------- 1/2 [pymanopt]
   ---------------------------------------- 2/2 [pymanopt]



In [ ]:
import time
import urllib.request
import urllib.error
import numpy as np
import networkx as nx
import cvxpy as cp

# Riemannian Optimization Libraries
try:
    import autograd.numpy as anp
    import pymanopt
    from pymanopt.manifolds import Oblique
    from pymanopt.optimizers import TrustRegions
    PYMANOPT_AVAILABLE = True
except ImportError:
    print("WARNING: 'pymanopt' or 'autograd' not installed. Burer-Monteiro route will fail.")
    PYMANOPT_AVAILABLE = False

# =========================
# 1. Online Benchmark Fetcher
# =========================
def fetch_gset_graph(gset_id="G1"):
    url = f"https://web.stanford.edu/~yyye/yyye/Gset/{gset_id}"
    print(f"Fetching {gset_id} from Stanford repository...")
    
    try:
        req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})
        response = urllib.request.urlopen(req)
        lines = response.read().decode('utf-8').splitlines()
    except Exception as e:
        print(f"Failed to download {gset_id}: {e}")
        return None

    first_line = lines[0].strip().split()
    n = int(first_line[0])
    
    G = nx.Graph()
    G.add_nodes_from(range(1, n + 1)) 
    
    for line in lines[1:]:
        parts = line.strip().split()
        if len(parts) >= 3:
            u, v, w = int(parts[0]), int(parts[1]), int(parts[2])
            G.add_edge(u, v, weight=w)
            
    G.name = f"GSET_{gset_id}"
    return G

# =========================
# 2. Graph Generator
# =========================
def generate_graphs(configs):
    graphs = []
    
    for idx, config in enumerate(configs):
        g_type = config.get('type')
        n = config.get('n', 50)
        seed = config.get('seed', 42 + idx)
        
        G = None 
        
        if g_type == 'gset':
            G = fetch_gset_graph(config.get('id', 'G1'))
            if G is not None: graphs.append(G)
            continue
        elif g_type == 'erdos_renyi': G = nx.erdos_renyi_graph(n, config.get('p', 0.3), seed=seed)
        elif g_type == 'barabasi_albert': G = nx.barabasi_albert_graph(n, config.get('m', 3), seed=seed)
        elif g_type == 'watts_strogatz': G = nx.watts_strogatz_graph(n, config.get('k', 4), config.get('p', 0.1), seed=seed)
        elif g_type == 'regular': G = nx.random_regular_graph(config.get('d', 3), n, seed=seed)
        elif g_type == 'powerlaw_cluster': G = nx.powerlaw_cluster_graph(n, config.get('m', 3), config.get('p', 0.1), seed=seed)
        elif g_type == 'geometric': G = nx.random_geometric_graph(n, config.get('radius', 0.2), seed=seed)
        elif g_type == 'complete': G = nx.complete_graph(n)
        elif g_type == 'complete_bipartite': G = nx.complete_bipartite_graph(config.get('n1', n // 2), config.get('n2', n - (n // 2)))
        elif g_type == 'cycle': G = nx.cycle_graph(n)
        elif g_type == 'star': G = nx.star_graph(n - 1)
        elif g_type == 'wheel': G = nx.wheel_graph(n)
        elif g_type == 'circular_ladder': G = nx.circular_ladder_graph(n // 2)
        elif g_type == 'grid_2d': G = nx.grid_2d_graph(config.get('dim1', int(np.sqrt(n))), config.get('dim2', int(np.sqrt(n))))
        elif g_type == 'hexagonal': G = nx.hexagonal_lattice_graph(config.get('m', int(np.sqrt(n))), config.get('m', int(np.sqrt(n))))
        elif g_type == 'hypercube': G = nx.hypercube_graph(config.get('dim', max(1, int(np.log2(n) if n > 0 else 3))))
        elif g_type == 'random_tree': G = nx.random_tree(n, seed=seed)
        elif g_type == 'balanced_tree': G = nx.balanced_tree(config.get('r', 2), config.get('h', 3))
        elif g_type == 'karate_club': G = nx.karate_club_graph()
        elif g_type == 'florentine_families': G = nx.florentine_families_graph()
        elif g_type == 'les_miserables': G = nx.les_miserables_graph()
        else:
            print(f"Skipping unknown graph type: '{g_type}'")
            continue

        if G is not None:
            G = nx.convert_node_labels_to_integers(G)
            G.name = f"{g_type.upper()}_V{G.number_of_nodes()}_E{G.number_of_edges()}_{idx}"
            for u, v, data in G.edges(data=True):
                if 'weight' not in data:
                    G[u][v]['weight'] = 1.0
            graphs.append(G)
            
    return graphs

# =========================
# 3. Traditional SDP (CVXPY)
# =========================
def solve_sdp_cvxpy(G):
    n = G.number_of_nodes()
    X = cp.Variable((n, n), symmetric=True)
    constraints = [X >> 0, cp.diag(X) == 1]
    L = nx.laplacian_matrix(G, weight='weight').toarray()
    
    prob = cp.Problem(cp.Maximize(cp.trace(L @ X) / 4), constraints)
    prob.solve(solver=cp.SCS)
    
    eigvals, eigvecs = np.linalg.eigh(X.value)
    eigvals = np.clip(eigvals, 0, None)
    V = eigvecs @ np.diag(np.sqrt(eigvals))
    return V, prob.value

# =========================
# 4. Burer-Monteiro SDP
# =========================
def solve_burer_monteiro(G, rank=15):
    n = G.number_of_nodes()
    L = nx.laplacian_matrix(G, weight='weight').toarray()
    manifold = Oblique(rank, n)
    
    @pymanopt.function.autograd(manifold)
    def cost(Y): return -0.25 * anp.trace(Y @ L @ Y.T)

    problem = pymanopt.Problem(manifold, cost)
    optimizer = TrustRegions(verbosity=0)
    Y_opt = optimizer.run(problem).point
    
    V = Y_opt.T
    sdp_val = 0.25 * np.trace(Y_opt @ L @ Y_opt.T)
    return V, sdp_val

# =========================
# 5. Rounding Methods
# =========================

def gw_rounding(G, V, **kwargs):
    """Classic Goemans-Williamson random hyperplane rounding."""
    num_samples = kwargs.get('num_samples', 2000)
    n, d = V.shape
    edges = list(G.edges(data='weight', default=1.0))
    edge_idx = np.array([[u, v] for u, v, w in edges])
    node_to_idx = {node: i for i, node in enumerate(G.nodes())}
    mapped_edges = np.array([[node_to_idx[u], node_to_idx[v]] for u, v in edge_idx])
    weights = np.array([w for u, v, w in edges])

    R = np.random.randn(d, num_samples)
    R /= np.linalg.norm(R, axis=0)
    signs = np.sign(V @ R)
    signs[signs == 0] = 1 

    cut_mask = signs[mapped_edges[:, 0]] != signs[mapped_edges[:, 1]]
    cuts = np.sum(cut_mask * weights[:, np.newaxis], axis=0)
    return cuts[np.argmax(cuts)]

def pca_rounding(G, V, **kwargs):
    """
    Deterministic alternative. Projects the high-dimensional vectors onto 
    their first Principal Component, splitting them by the hyperplane 
    orthogonal to the direction of maximum variance.
    """
    n, d = V.shape
    edges = list(G.edges(data='weight', default=1.0))
    edge_idx = np.array([[u, v] for u, v, w in edges])
    node_to_idx = {node: i for i, node in enumerate(G.nodes())}
    mapped_edges = np.array([[node_to_idx[u], node_to_idx[v]] for u, v in edge_idx])
    weights = np.array([w for u, v, w in edges])

    # Singular Value Decomposition to find the principal component
    _, _, Vh = np.linalg.svd(V, full_matrices=False)
    pc1 = Vh[0, :] # First principal component
    
    # Project and extract signs
    signs = np.sign(V @ pc1)
    signs[signs == 0] = 1
    
    # Calculate cut for this single deterministic partition
    cut_mask = signs[mapped_edges[:, 0]] != signs[mapped_edges[:, 1]]
    return np.sum(cut_mask * weights)

# --- Rounding Registry ---
ROUNDING_METHODS = {
    'gw': gw_rounding,
    'pca': pca_rounding
}

# =========================
# 6. Fallback: Fast Local Search
# =========================
def greedy_local_search(G, num_restarts=10):
    best_overall_cut = -np.inf
    nodes = list(G.nodes())

    for _ in range(num_restarts):
        partition = {node: np.random.choice([1, -1]) for node in nodes}
        improved = True
        
        while improved:
            improved = False
            for u in nodes:
                gain = sum(
                    G[u][v].get('weight', 1.0) if partition[u] == partition[v] else -G[u][v].get('weight', 1.0)
                    for v in G.neighbors(u)
                )
                if gain > 0:
                    partition[u] *= -1
                    improved = True

        current_cut = sum(data.get('weight', 1.0) for u, v, data in G.edges(data=True) if partition[u] != partition[v])
        if current_cut > best_overall_cut:
            best_overall_cut = current_cut

    return best_overall_cut

# =========================
# 7. Smart Pipeline Router
# =========================
def evaluate_maxcut(G, cvxpy_threshold=500, pymanopt_threshold=5000, rounding_config=None):
    """
    Evaluates MaxCut on G. Uses rounding_config to dynamically select post-SDP rounding method.
    """
    if rounding_config is None:
        rounding_config = {'method': 'gw', 'kwargs': {'num_samples': 2000}}
        
    rounding_name = rounding_config.get('method', 'gw')
    rounding_kwargs = rounding_config.get('kwargs', {})
    
    # Fetch the function from our registry, fallback to 'gw' if not found
    rounding_func = ROUNDING_METHODS.get(rounding_name, gw_rounding)

    n = G.number_of_nodes()
    t0 = time.perf_counter()

    if n <= cvxpy_threshold:
        print(f"  -> Routing to CVXPY (N={n} <= {cvxpy_threshold})")
        V, sdp_val = solve_sdp_cvxpy(G)
        cut = rounding_func(G, V, **rounding_kwargs)
        method = f"CVXPY + {rounding_name.upper()} Rounding"

    elif n <= pymanopt_threshold and PYMANOPT_AVAILABLE:
        print(f"  -> Routing to PyManopt Burer-Monteiro (N={n})")
        V, sdp_val = solve_burer_monteiro(G, rank=15)
        cut = rounding_func(G, V, **rounding_kwargs)
        method = f"PyManopt BM + {rounding_name.upper()} Rounding"

    else:
        print(f"  -> Routing to Greedy Heuristic (N={n} > {pymanopt_threshold})")
        cut = greedy_local_search(G, num_restarts=15)
        sdp_val = None
        method = "GREEDY_SEARCH"

    total_time = time.perf_counter() - t0
    approx_ratio = cut / sdp_val if sdp_val else None

    return {
        "graph_name": getattr(G, "name", "Unnamed"),
        "method": method,
        "nodes": n,
        "edges": G.number_of_edges(),
        "sdp_val": sdp_val,
        "cut_val": cut,
        "approx_ratio": approx_ratio,
        "total_time": total_time
    }

# =========================
# RUN BATCH EXPERIMENTS
# =========================
if __name__ == "__main__":
    experiment_configs = [
        {'type': 'erdos_renyi', 'n': 100, 'p': 0.1},
        {'type': 'karate_club'},
        {'type': 'complete_bipartite', 'n1': 50, 'n2': 50}
    ]
    
    print("Generating/Fetching graphs...")
    graphs = generate_graphs(experiment_configs)
    
    print("\nStarting Benchmark Pipeline...\n" + "="*65)
    
    # We will test two different rounding configurations for each graph
    rounding_tests = [
        {'method': 'gw', 'kwargs': {'num_samples': 2000}},
        {'method': 'pca', 'kwargs': {}}  # PCA requires no extra kwargs
    ]
    
    for G in graphs:
        for r_conf in rounding_tests:
            print(f"Solving {G.name} with {r_conf['method'].upper()} rounding...")
            
            res = evaluate_maxcut(G, rounding_config=r_conf)
            
            print(f"  ├─ Method Used:     {res['method']}")
            if res['sdp_val']:
                print(f"  ├─ SDP Upper Bound: {res['sdp_val']:.2f}")
                print(f"  ├─ Approx Ratio:    {res['approx_ratio']:.4f}")
            else:
                print(f"  ├─ SDP Upper Bound: Skipped")
                
            print(f"  ├─ Actual Cut:      {res['cut_val']}")
            print(f"  └─ Total Time:      {res['total_time']:.3f}s\n")
        print("-" * 65)

Generating/Fetching graphs...
Fetching G1 from Stanford repository...
Fetching G22 from Stanford repository...

Starting Benchmark Pipeline...
Solving ERDOS_RENYI_n100_0 (|V|=100, |E|=474)...
  -> Routing to CVXPY (N=100 <= 500)
  ├─ Method Used:     CVXPY + GW
  ├─ SDP Upper Bound: 358.66
  ├─ Approx Ratio:    0.9424
  ├─ Actual Cut:      338.0
  └─ Total Time:      0.259s

-----------------------------------------------------------------
Solving GSET_G1 (|V|=800, |E|=19176)...
  -> Routing to PyManopt Burer-Monteiro (N=800)
  ├─ Method Used:     PyManopt BM + GW
  ├─ SDP Upper Bound: 12083.20
  ├─ Approx Ratio:    0.9444
  ├─ Actual Cut:      11411
  └─ Total Time:      7.196s

-----------------------------------------------------------------
Solving GSET_G22 (|V|=2000, |E|=19990)...
  -> Routing to PyManopt Burer-Monteiro (N=2000)
  ├─ Method Used:     PyManopt BM + GW
  ├─ SDP Upper Bound: 14135.66
  ├─ Approx Ratio:    0.9181
  ├─ Actual Cut:      12978
  └─ Total Time:      36.09